# Escalón 2 · mmBERT-small jerárquico

Ajuste fino de `jhu-clsp/mmBERT-small` (MIT) con dos cabezas: subclase de 5 dígitos y código de 9 dígitos. Se evalúa con **el mismo protocolo** que la línea base (`cpcmini.pipeline.evaluate_model`): temperatura y umbral en validación 2024, métricas en prueba 2025–2026.

Requisitos: GPU T4 o P100 de Kaggle (confirma tu cuota semanal en la cuenta). Al terminar, descarga `reports/metrics/mmbert_small.json` y cópialo en `reports/metrics/` del repositorio; `cpcmini report` lo incluirá en la tabla.

In [ ]:
# Entorno: el repositorio público y sus dependencias de GPU (torch ya viene en Kaggle).
!git clone --depth 1 https://github.com/DiegoFranciscoG/cpc-mini.git
%cd cpc-mini
!pip install -q -e ".[gpu]"

In [ ]:
# Datos: sube data/processed (train, val, test, catalog y manifest) como dataset PRIVADO de Kaggle
# llamado "cpc-mini-processed". El cuaderno no descarga nada del SERCOP.
import json, time
from pathlib import Path
from cpcmini.config import load_settings
from cpcmini.data.dataset import read_manifest, read_split
from cpcmini.pipeline import cap_per_class, evaluate_model, write_report

PROCESSED = "/kaggle/input/cpc-mini-processed"
OUT = Path("/kaggle/working")
settings = load_settings("configs")
manifest = read_manifest(PROCESSED)
print("dataset", manifest["dataset_hash"][:12])
train = cap_per_class(read_split(PROCESSED, "train"), settings.models.linear.max_per_class)
print(train.height, "descripciones de entrenamiento")

In [ ]:
# Entrenamiento (semilla fija de configs/models.yaml).
from cpcmini.models.encoder import EncoderClassifier, train_encoder

start = time.perf_counter()
folder = train_encoder(train["desc_norm"].to_list(), train["code9"].to_list(),
                       settings.models.encoder, OUT / "mmbert_small", settings.models.seed)
train_seconds = time.perf_counter() - start
print(f"{train_seconds/60:.1f} min")

In [ ]:
# Evaluación con el protocolo común.
model = EncoderClassifier(folder, max_length=settings.models.encoder.max_length)
metrics, temperature, threshold = evaluate_model(model, "mmbert_small", settings, PROCESSED,
                                                 train.height, train_seconds)
write_report(metrics, OUT / "reports")
print(json.dumps({k: metrics["test"][k]["top1"] for k in ("level_5", "level_9")}, indent=1))

## T-06 · Reproducibilidad

Vuelve a ejecutar las dos celdas anteriores en una sesión nueva: con la misma semilla y el mismo `dataset_hash`, el top-1 debe coincidir dentro de `reproducibility_tolerance` (0,001). En GPU algunas operaciones no son deterministas; si la diferencia supera la tolerancia, se reporta tal cual.

## Exportación a CPU (sólo si gana)

Si este escalón gana en validación, expórtalo a ONNX con `optimum` y cuantízalo a int8; la prueba T-07 exige perder menos de 2 puntos frente al modelo en float32.